# GCON Model Input - Horizon 2 Model Comparison

Notebook n?y kh?p v?i pipeline GCON hi?n t?i:
- Input: `cleaned_data/gcon_customer_month_clean.parquet`, ???c t?o t? notebook GCON cleaning/merge.
- Target/EDA logic: gi?ng `GCON_EDA_Subscription_Inline.ipynb`, g?m filter `CUM_SUBSCRIPTION_BEFORE == 0` ?? ch? gi? observation tr??c l?n subscription ??u ti?n cho t?ng customer-product.
- Output: model-ready data, train/test split theo th?i gian, train v? so s?nh 2 model: LightGBM v? XGBoost.

M?c ti?u l? propensity/ranking cho NBFO, kh?ng d?ng SMOTE m?c ??nh ?? tr?nh l?m l?ch calibration.

In [10]:
from pathlib import Path
import warnings
import time

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 120)

CLEAN_DIR = Path('cleaned_data')
OUTPUT_DIR = Path('model_data_gcon')
OUTPUT_DIR.mkdir(exist_ok=True)

HORIZON = 2
TEST_START_MONTH = None  # None = last month that has full future horizon labels.
RANDOM_STATE = 42
USE_SAMPLE_FOR_FAST_RUN = False
SAMPLE_ROWS = 400_000

PRODUCTS = pd.DataFrame([
    {'PRODUCT_CODE': 101, 'PRODUCT_NAME': 'CURRENT_ACCOUNT', 'OWN_COL': 'OWN_CURRENT_ACCOUNT'},
    {'PRODUCT_CODE': 102, 'PRODUCT_NAME': 'TERM_DEPOSIT', 'OWN_COL': 'OWN_TERM_DEPOSIT'},
    {'PRODUCT_CODE': 103, 'PRODUCT_NAME': 'CREDIT_CARD', 'OWN_COL': 'OWN_CREDIT_CARD'},
    {'PRODUCT_CODE': 104, 'PRODUCT_NAME': 'DEBIT_CARD', 'OWN_COL': 'OWN_DEBIT_CARD'},
    {'PRODUCT_CODE': 105, 'PRODUCT_NAME': 'LENDING', 'OWN_COL': 'OWN_LENDING'},
])
PRODUCTS

,PRODUCT_CODE,PRODUCT_NAME,OWN_COL
0,101,CURRENT_ACCOUNT,OWN_CURRENT_ACCOUNT
1,102,TERM_DEPOSIT,OWN_TERM_DEPOSIT
2,103,CREDIT_CARD,OWN_CREDIT_CARD
3,104,DEBIT_CARD,OWN_DEBIT_CARD
4,105,LENDING,OWN_LENDING


## 1. Helper Functions

In [11]:
def ensure_numeric(df, cols):
    for col in cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df


def save_table(df, path_stem):
    parquet_path = OUTPUT_DIR / f'{path_stem}.parquet'
    csv_path = OUTPUT_DIR / f'{path_stem}.csv'
    try:
        df.to_parquet(parquet_path, index=False)
        print('saved', parquet_path, df.shape)
    except Exception as exc:
        df.to_csv(csv_path, index=False)
        print('saved', csv_path, df.shape, '| parquet failed:', type(exc).__name__)


def safe_to_csv(df, path):
    path = Path(path)
    try:
        df.to_csv(path, index=False)
        print('saved', path, df.shape)
    except PermissionError:
        fallback = path.with_name(f'{path.stem}_new{path.suffix}')
        df.to_csv(fallback, index=False)
        print('permission denied for', path, '| saved fallback', fallback, df.shape)


def add_customer_month_features(base):
    df = base.sort_values(['CUSTOMER_NUMBER', 'MONTH']).copy()
    numeric_defaults = [
        'AVG_CA_BALANCE', 'AVG_TD_BALANCE', 'COUNT_CA_ACCT', 'COUNT_TD_ACCT',
        'COUNT_CREDITCARD', 'COUNT_DEBITCARD', 'OVERDUE_CREDIT', 'LIMIT_AMT_CREDIT', 'OUTSTANDING_BAL_CREDIT',
        'COUNT_OF_LOAN', 'AVG_LOAN_AMOUNT', 'OVERDUE_LENDING', 'TERM_LENDING', 'INTEREST_RATE',
        'TRANS_RECORDS', 'TRANS_NO_SUM', 'TRANS_AMOUNT_SUM', 'TRANS_AMOUNT_MEAN', 'TRANS_AMOUNT_MAX',
        'TRANS_ACTIVE_DAYS', 'TRANS_TYPE_LV1_NUNIQUE', 'TRANS_TYPE_LV2_NUNIQUE', 'DEVICE_NUNIQUE', 'MERCHANT_NUNIQUE',
        'INTERNAL_TRANSFER_ROWS', 'ACTIVITY_RECORDS', 'ACTIVITY_NO_SUM', 'ACTIVITY_ACTIVE_DAYS',
        'ACTIVITY_NAME_NUNIQUE', 'ACTIVITY_HOUR_NUNIQUE', 'AGE'
    ]
    predictive_activity_types = [
        'ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_INFORMATION',
        'ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_PORFOLIO',
        'ACTIVITY_TYPE_COUNT_QUERY_CURRENT_ACCOUNT',
        'ACTIVITY_TYPE_COUNT_TRANSACTION_DETAIL_QUERY',
        'ACTIVITY_TYPE_COUNT_TRANSACTION_OVERVIEW_QUERY',
        'ACTIVITY_TYPE_COUNT_TRANSFER_BANK_ACCOUNT',
        'ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PHONENO',
        'ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PAYMENT_CENTER',
        'ACTIVITY_TYPE_COUNT_RB_BILLPAY_MOBILE',
        'ACTIVITY_TYPE_COUNT_TOPUP_MOBILE',
        'ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_CASHBACK',
        'ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_REDEEM',
        'ACTIVITY_TYPE_COUNT_EXPORT_ACCOUNT_STATEMENT_LOAN',
    ]
    all_activity_type_count_cols = sorted([c for c in df.columns if c.startswith('ACTIVITY_TYPE_COUNT_')])
    activity_type_count_cols = [c for c in predictive_activity_types if c in df.columns]
    df = df.drop(columns=[c for c in all_activity_type_count_cols if c not in activity_type_count_cols])
    df = ensure_numeric(df, numeric_defaults + activity_type_count_cols)

    df['OWN_CURRENT_ACCOUNT'] = (df['COUNT_CA_ACCT'] > 0).astype('int8')
    df['OWN_TERM_DEPOSIT'] = (df['COUNT_TD_ACCT'] > 0).astype('int8')
    df['OWN_CREDIT_CARD'] = (df['COUNT_CREDITCARD'] > 0).astype('int8')
    df['OWN_DEBIT_CARD'] = (df['COUNT_DEBITCARD'] > 0).astype('int8')
    df['OWN_LENDING'] = ((df['COUNT_OF_LOAN'] > 0) | (df['AVG_LOAN_AMOUNT'] > 0)).astype('int8')

    own_cols = PRODUCTS['OWN_COL'].tolist()
    df['SPTC_COUNT'] = df[own_cols].sum(axis=1).astype('int8')
    df['HAS_MULTI_SPTC'] = (df['SPTC_COUNT'] >= 2).astype('int8')
    df['TOTAL_DEPOSIT_BALANCE'] = df['AVG_CA_BALANCE'] + df['AVG_TD_BALANCE']
    df['TOTAL_DEPOSIT_ACCTS'] = df['COUNT_CA_ACCT'] + df['COUNT_TD_ACCT']
    df['CARD_UTILIZATION'] = np.where(
        df['LIMIT_AMT_CREDIT'] > 0,
        df['OUTSTANDING_BAL_CREDIT'] / df['LIMIT_AMT_CREDIT'],
        0,
    )
    df['CARD_UTILIZATION'] = np.clip(df['CARD_UTILIZATION'], 0, 5)
    df['HAS_TRANSACTION'] = (df['TRANS_RECORDS'] > 0).astype('int8')
    df['HAS_ACTIVITY'] = (df['ACTIVITY_RECORDS'] > 0).astype('int8')
    df['ACTIVITY_EVENTS_PER_ACTIVE_DAY'] = np.where(
        df['ACTIVITY_ACTIVE_DAYS'] > 0,
        df['ACTIVITY_RECORDS'] / df['ACTIVITY_ACTIVE_DAYS'],
        0,
    )
    df['ACTIVITY_TYPES_PER_EVENT'] = np.where(
        df['ACTIVITY_RECORDS'] > 0,
        df['ACTIVITY_NAME_NUNIQUE'] / df['ACTIVITY_RECORDS'],
        0,
    )
    df['ACTIVITY_HOURS_PER_ACTIVE_DAY'] = np.where(
        df['ACTIVITY_ACTIVE_DAYS'] > 0,
        df['ACTIVITY_HOUR_NUNIQUE'] / df['ACTIVITY_ACTIVE_DAYS'],
        0,
    )
    df['AGE_CLEAN'] = df['AGE'].where(df['AGE'].between(16, 100))
    df['AGE_GROUP'] = pd.cut(
        df['AGE_CLEAN'],
        bins=[0, 25, 35, 45, 55, 65, 200],
        labels=['<=25', '26-35', '36-45', '46-55', '56-65', '65+'],
    )
    df['CUSTOMER_TENURE_MONTHS'] = (
        (df['MONTH'] - pd.to_datetime(df['CLIENT_CREATE_DATE'], errors='coerce')).dt.days / 30.4375
    ).clip(lower=0)
    df['IB_TENURE_MONTHS'] = (
        (df['MONTH'] - pd.to_datetime(df['IB_REGISTER_DATE'], errors='coerce')).dt.days / 30.4375
    ).clip(lower=0)
    df['CUSTOMER_TENURE_BIN'] = pd.cut(
        df['CUSTOMER_TENURE_MONTHS'],
        bins=[-1, 3, 6, 12, 24, 48, 10000],
        labels=['<3m', '3-6m', '6-12m', '1-2y', '2-4y', '4y+'],
    )
    df['ACTIVITY_SEGMENT'] = pd.cut(
        df['TRANS_RECORDS'],
        bins=[-1, 0, 5, 20, 10000000],
        labels=['No transaction', 'Low', 'Medium', 'High'],
    )

    for col in ['TOTAL_DEPOSIT_BALANCE', 'AVG_CA_BALANCE', 'AVG_TD_BALANCE', 'AVG_LOAN_AMOUNT',
                'TRANS_AMOUNT_SUM', 'TRANS_AMOUNT_MEAN', 'TRANS_AMOUNT_MAX', 'ACTIVITY_NO_SUM']:
        df[f'LOG_{col}'] = np.log1p(df[col].clip(lower=0))

    grp = df.groupby('CUSTOMER_NUMBER', sort=False)
    lag_cols = ['TOTAL_DEPOSIT_BALANCE', 'TOTAL_DEPOSIT_ACCTS', 'TRANS_AMOUNT_SUM', 'TRANS_RECORDS',
                'TRANS_ACTIVE_DAYS', 'ACTIVITY_RECORDS', 'ACTIVITY_ACTIVE_DAYS',
                'ACTIVITY_NAME_NUNIQUE', 'SPTC_COUNT']
    for col in lag_cols:
        df[f'{col}_LAG1'] = grp[col].shift(1).fillna(0)
        df[f'{col}_DIFF1'] = df[col] - df[f'{col}_LAG1']
        df[f'{col}_ROLL3_MEAN'] = (
            grp[col]
            .shift(1)
            .groupby(df['CUSTOMER_NUMBER'], sort=False)
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
            .fillna(0)
        )

    # Banking RFM features from monthly customer history.
    # Recency is approximated from monthly aggregates: months since last active month * 30.4375.
    month_index = df['MONTH'].dt.year * 12 + df['MONTH'].dt.month
    df['_MONTH_INDEX'] = month_index

    active_txn_month = df['_MONTH_INDEX'].where(df['TRANS_RECORDS'] > 0)
    active_login_month = df['_MONTH_INDEX'].where(df['ACTIVITY_RECORDS'] > 0)
    product_open_flag = df.groupby('CUSTOMER_NUMBER', sort=False)[own_cols].diff().gt(0).any(axis=1)
    product_open_month = df['_MONTH_INDEX'].where(product_open_flag)

    df['_LAST_TXN_MONTH_BEFORE'] = active_txn_month.groupby(df['CUSTOMER_NUMBER']).ffill().groupby(df['CUSTOMER_NUMBER']).shift(1)
    df['_LAST_LOGIN_MONTH_BEFORE'] = active_login_month.groupby(df['CUSTOMER_NUMBER']).ffill().groupby(df['CUSTOMER_NUMBER']).shift(1)
    df['_LAST_PRODUCT_OPEN_MONTH_BEFORE'] = product_open_month.groupby(df['CUSTOMER_NUMBER']).ffill().groupby(df['CUSTOMER_NUMBER']).shift(1)

    df['DAYS_SINCE_LAST_TRANSACTION'] = ((df['_MONTH_INDEX'] - df['_LAST_TXN_MONTH_BEFORE']) * 30.4375).fillna(9999).clip(lower=0)
    df['DAYS_SINCE_LAST_LOGIN'] = ((df['_MONTH_INDEX'] - df['_LAST_LOGIN_MONTH_BEFORE']) * 30.4375).fillna(9999).clip(lower=0)
    df['DAYS_SINCE_LAST_PRODUCT_OPEN'] = ((df['_MONTH_INDEX'] - df['_LAST_PRODUCT_OPEN_MONTH_BEFORE']) * 30.4375).fillna(9999).clip(lower=0)

    df['TXN_COUNT_LAST_30D'] = df['TRANS_RECORDS']
    df['TXN_COUNT_LAST_90D'] = (
        grp['TRANS_RECORDS']
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )
    df['ACTIVE_DAYS_LAST_90D'] = (
        grp['TRANS_ACTIVE_DAYS']
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )
    df['AVG_MONTHLY_TXN'] = (
        grp['TRANS_RECORDS']
        .expanding(min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )

    # Activity frequency features: aggregate intensity, history, and selected predictive activity-type cadence.
    df['ACTIVITY_COUNT_LAST_30D'] = df['ACTIVITY_RECORDS']
    df['ACTIVITY_COUNT_LAST_90D'] = (
        grp['ACTIVITY_RECORDS']
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )
    df['ACTIVITY_ACTIVE_DAYS_LAST_90D'] = (
        grp['ACTIVITY_ACTIVE_DAYS']
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )
    df['AVG_MONTHLY_ACTIVITY'] = (
        grp['ACTIVITY_RECORDS']
        .expanding(min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )
    df['ACTIVITY_ACTIVE_MONTHS_HISTORY'] = (
        grp['HAS_ACTIVITY']
        .cumsum()
        .groupby(df['CUSTOMER_NUMBER'])
        .shift(fill_value=0)
        .astype('int16')
    )
    df['ACTIVITY_ACTIVE_MONTH_RATE_HISTORY'] = (
        grp['HAS_ACTIVITY']
        .expanding(min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .groupby(df['CUSTOMER_NUMBER'])
        .shift(fill_value=0)
    )
    df['ACTIVITY_RECORDS_PER_ACTIVE_DAY_90D'] = np.where(
        df['ACTIVITY_ACTIVE_DAYS_LAST_90D'] > 0,
        df['ACTIVITY_COUNT_LAST_90D'] / df['ACTIVITY_ACTIVE_DAYS_LAST_90D'],
        0,
    )
    for col in activity_type_count_cols:
        df[f'{col}_SHARE'] = np.where(df['ACTIVITY_RECORDS'] > 0, df[col] / df['ACTIVITY_RECORDS'], 0)
        df[f'{col}_LAST_90D'] = (
            grp[col]
            .rolling(3, min_periods=1)
            .sum()
            .reset_index(level=0, drop=True)
            .fillna(0)
        )
        df[f'{col}_AVG_90D'] = (
            grp[col]
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
            .fillna(0)
        )

    df['AVG_BALANCE'] = df['TOTAL_DEPOSIT_BALANCE']
    df['TOTAL_DEPOSIT'] = df['TOTAL_DEPOSIT_BALANCE']
    df['TOTAL_TRANSACTION_AMOUNT'] = df['TRANS_AMOUNT_SUM']
    df['TOTAL_TRANSACTION_AMOUNT_90D'] = (
        grp['TRANS_AMOUNT_SUM']
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )
    df['AVG_BALANCE_90D'] = (
        grp['TOTAL_DEPOSIT_BALANCE']
        .rolling(3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )

    # Product transition history, recency by product, and frequency by product.
    product_history_specs = {
        'CURRENT_ACCOUNT': {'own': 'OWN_CURRENT_ACCOUNT', 'count': 'COUNT_CA_ACCT'},
        'TERM_DEPOSIT': {'own': 'OWN_TERM_DEPOSIT', 'count': 'COUNT_TD_ACCT'},
        'CREDIT_CARD': {'own': 'OWN_CREDIT_CARD', 'count': 'COUNT_CREDITCARD'},
        'DEBIT_CARD': {'own': 'OWN_DEBIT_CARD', 'count': 'COUNT_DEBITCARD'},
        'LENDING': {'own': 'OWN_LENDING', 'count': 'COUNT_OF_LOAN'},
    }
    product_open_cols = []
    for product_name, spec in product_history_specs.items():
        own_col = spec['own']
        count_col = spec['count']
        open_col = f'PRODUCT_OPEN_FLAG_{product_name}'
        product_open_cols.append(open_col)
        df[open_col] = df.groupby('CUSTOMER_NUMBER', sort=False)[own_col].diff().gt(0).astype('int8')
        first_obs_open = (df.groupby('CUSTOMER_NUMBER', sort=False).cumcount() == 0) & (df[own_col] == 1)
        df.loc[first_obs_open, open_col] = 1

        open_month = df['_MONTH_INDEX'].where(df[open_col] == 1)
        last_open_before = open_month.groupby(df['CUSTOMER_NUMBER']).ffill().groupby(df['CUSTOMER_NUMBER']).shift(1)
        df[f'DAYS_SINCE_LAST_OPEN_{product_name}'] = ((df['_MONTH_INDEX'] - last_open_before) * 30.4375).fillna(9999).clip(lower=0)
        df[f'{product_name}_OPEN_COUNT_HISTORY'] = (
            df.groupby('CUSTOMER_NUMBER', sort=False)[open_col]
            .cumsum()
            .groupby(df['CUSTOMER_NUMBER'])
            .shift(fill_value=0)
            .astype('int16')
        )
        df[f'{product_name}_OWNED_MONTHS_HISTORY'] = (
            df.groupby('CUSTOMER_NUMBER', sort=False)[own_col]
            .cumsum()
            .groupby(df['CUSTOMER_NUMBER'])
            .shift(fill_value=0)
            .astype('int16')
        )
        df[f'{product_name}_OWN_RATE_HISTORY'] = (
            df.groupby('CUSTOMER_NUMBER', sort=False)[own_col]
            .expanding(min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
            .groupby(df['CUSTOMER_NUMBER'])
            .shift(fill_value=0)
        )
        df[f'{product_name}_COUNT_LAST_30D'] = df[count_col]
        df[f'{product_name}_COUNT_LAST_90D'] = (
            df.groupby('CUSTOMER_NUMBER', sort=False)[count_col]
            .rolling(3, min_periods=1)
            .sum()
            .reset_index(level=0, drop=True)
            .fillna(0)
        )
        df[f'{product_name}_COUNT_AVG_90D'] = (
            df.groupby('CUSTOMER_NUMBER', sort=False)[count_col]
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
            .fillna(0)
        )

    df['PRODUCT_OPEN_COUNT_HISTORY'] = df[[f'{p}_OPEN_COUNT_HISTORY' for p in product_history_specs]].sum(axis=1)
    df['PRODUCT_TRANSITION_COUNT_LAST_90D'] = (
        df.groupby('CUSTOMER_NUMBER', sort=False)[product_open_cols]
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .sum(axis=1)
        .fillna(0)
    )

    for col in ['TXN_COUNT_LAST_90D', 'TOTAL_TRANSACTION_AMOUNT_90D', 'AVG_BALANCE_90D', 'DAYS_SINCE_LAST_TRANSACTION', 'DAYS_SINCE_LAST_LOGIN', 'DAYS_SINCE_LAST_PRODUCT_OPEN', 'ACTIVITY_COUNT_LAST_90D', 'ACTIVITY_ACTIVE_DAYS_LAST_90D', 'AVG_MONTHLY_ACTIVITY', 'ACTIVITY_RECORDS_PER_ACTIVE_DAY_90D']:
        df[f'LOG_{col}'] = np.log1p(df[col].clip(lower=0))
    for col in activity_type_count_cols:
        df[f'LOG_{col}_LAST_90D'] = np.log1p(df[f'{col}_LAST_90D'].clip(lower=0))
    for product_name in product_history_specs:
        df[f'LOG_DAYS_SINCE_LAST_OPEN_{product_name}'] = np.log1p(df[f'DAYS_SINCE_LAST_OPEN_{product_name}'].clip(lower=0))
        df[f'LOG_{product_name}_COUNT_LAST_90D'] = np.log1p(df[f'{product_name}_COUNT_LAST_90D'].clip(lower=0))

    df = df.drop(columns=[
        '_MONTH_INDEX', '_LAST_TXN_MONTH_BEFORE', '_LAST_LOGIN_MONTH_BEFORE', '_LAST_PRODUCT_OPEN_MONTH_BEFORE'
    ])
    return df.replace([np.inf, -np.inf], np.nan)


def make_subscription_long(feature_df):
    own_cols = PRODUCTS['OWN_COL'].tolist()
    keys = feature_df[['CUSTOMER_NUMBER', 'MONTH']].drop_duplicates().copy()
    keys['_key'] = 1
    products = PRODUCTS.copy()
    products['_key'] = 1
    long_keys = keys.merge(products, on='_key', how='outer').drop(columns='_key')

    own_long = feature_df[['CUSTOMER_NUMBER', 'MONTH'] + own_cols].melt(
        id_vars=['CUSTOMER_NUMBER', 'MONTH'],
        value_vars=own_cols,
        var_name='OWN_COL',
        value_name='OWNS_PRODUCT_T',
    )
    label_df = long_keys.merge(own_long, on=['CUSTOMER_NUMBER', 'MONTH', 'OWN_COL'], how='left')
    label_df['OWNS_PRODUCT_T'] = label_df['OWNS_PRODUCT_T'].fillna(0).astype('int8')

    future_cols = []
    for step in range(1, HORIZON + 1):
        tmp = own_long.copy()
        tmp['MONTH'] = tmp['MONTH'] - pd.offsets.MonthEnd(step)
        col = f'OWNS_PRODUCT_T_PLUS_{step}'
        future_cols.append(col)
        tmp = tmp.rename(columns={'OWNS_PRODUCT_T': col})
        label_df = label_df.merge(tmp, on=['CUSTOMER_NUMBER', 'MONTH', 'OWN_COL'], how='left')

    label_df[future_cols] = label_df[future_cols].fillna(0).astype('int8')
    max_observed_month = feature_df['MONTH'].max()
    last_label_month = max_observed_month - pd.offsets.MonthEnd(HORIZON)
    label_df['HAS_FULL_LABEL_HORIZON'] = (label_df['MONTH'] <= last_label_month).astype('int8')
    label_df['FUTURE_OWNS_PRODUCT'] = label_df[future_cols].max(axis=1).astype('int8')
    label_df['SUBSCRIPTION'] = ((label_df['OWNS_PRODUCT_T'] == 0) & (label_df['FUTURE_OWNS_PRODUCT'] == 1)).astype('int8')

    label_df = label_df.sort_values(['CUSTOMER_NUMBER', 'OWN_COL', 'MONTH']).reset_index(drop=True)
    label_df['CUM_SUBSCRIPTION_BEFORE'] = (
        label_df
        .groupby(['CUSTOMER_NUMBER', 'OWN_COL'])['SUBSCRIPTION']
        .transform(lambda s: s.cumsum().shift(fill_value=0))
        .astype('int16')
    )

    pre_rows = int(((label_df['OWNS_PRODUCT_T'] == 0) & (label_df['HAS_FULL_LABEL_HORIZON'] == 1)).sum())
    pre_pos = int(((label_df['OWNS_PRODUCT_T'] == 0) & (label_df['HAS_FULL_LABEL_HORIZON'] == 1) & (label_df['SUBSCRIPTION'] == 1)).sum())
    model_df = label_df.query(
        'OWNS_PRODUCT_T == 0 and HAS_FULL_LABEL_HORIZON == 1 and CUM_SUBSCRIPTION_BEFORE == 0'
    ).copy()
    print('pre first-sub filter rows/positives:', pre_rows, pre_pos)
    print('post first-sub filter rows/positives:', len(model_df), int(model_df['SUBSCRIPTION'].sum()))

    model_df = model_df.merge(feature_df, on=['CUSTOMER_NUMBER', 'MONTH'], how='left')

    # Target-product-specific history features.
    product_name_by_own_col = PRODUCTS.set_index('OWN_COL')['PRODUCT_NAME'].to_dict()
    product_count_col = {
        'CURRENT_ACCOUNT': 'COUNT_CA_ACCT',
        'TERM_DEPOSIT': 'COUNT_TD_ACCT',
        'CREDIT_CARD': 'COUNT_CREDITCARD',
        'DEBIT_CARD': 'COUNT_DEBITCARD',
        'LENDING': 'COUNT_OF_LOAN',
    }
    conditions = [model_df['OWN_COL'].eq(own_col) for own_col in product_name_by_own_col]
    product_names = [product_name_by_own_col[own_col] for own_col in product_name_by_own_col]

    def select_target_product_feature(template):
        choices = [model_df[template.format(product=product_name)] for product_name in product_names]
        return np.select(conditions, choices, default=0)

    model_df['TARGET_DAYS_SINCE_LAST_PRODUCT_OPEN'] = select_target_product_feature('DAYS_SINCE_LAST_OPEN_{product}')
    model_df['TARGET_PRODUCT_OPEN_COUNT_HISTORY'] = select_target_product_feature('{product}_OPEN_COUNT_HISTORY')
    model_df['TARGET_PRODUCT_OWNED_MONTHS_HISTORY'] = select_target_product_feature('{product}_OWNED_MONTHS_HISTORY')
    model_df['TARGET_PRODUCT_OWN_RATE_HISTORY'] = select_target_product_feature('{product}_OWN_RATE_HISTORY')
    model_df['TARGET_PRODUCT_COUNT_LAST_30D'] = np.select(
        conditions,
        [model_df[product_count_col[product_name]] for product_name in product_names],
        default=0,
    )
    model_df['TARGET_PRODUCT_COUNT_LAST_90D'] = select_target_product_feature('{product}_COUNT_LAST_90D')
    model_df['TARGET_PRODUCT_COUNT_AVG_90D'] = select_target_product_feature('{product}_COUNT_AVG_90D')
    model_df['LOG_TARGET_DAYS_SINCE_LAST_PRODUCT_OPEN'] = np.log1p(model_df['TARGET_DAYS_SINCE_LAST_PRODUCT_OPEN'].clip(lower=0))
    model_df['LOG_TARGET_PRODUCT_COUNT_LAST_90D'] = np.log1p(model_df['TARGET_PRODUCT_COUNT_LAST_90D'].clip(lower=0))
    return model_df


## 2. Load GCON Customer-Month Data

In [12]:
base_path = CLEAN_DIR / 'gcon_customer_month_clean.parquet'
if not base_path.exists():
    raise FileNotFoundError(f'Missing {base_path}. Run [GCON_26] cleaning/merge notebook first.')

base = pd.read_parquet(base_path)
base['MONTH'] = pd.to_datetime(base['MONTH'], errors='coerce').dt.to_period('M').dt.to_timestamp('M')
base['CUSTOMER_NUMBER'] = pd.to_numeric(base['CUSTOMER_NUMBER'], errors='coerce').astype('Int64')
base = base.dropna(subset=['CUSTOMER_NUMBER', 'MONTH']).copy()

# Match GCON EDA: only customers with IB registration.
base = base[base['IB_REGISTER_DATE'].notna()].copy()

print(base.shape)
print(base['MONTH'].min(), base['MONTH'].max())
display(base.head())

(889631, 89)
2019-01-31 00:00:00 2019-12-31 00:00:00


,CUSTOMER_NUMBER,MONTH,CLIENT_SEX,CLIENT_CREATE_DATE,DATE_OF_BIRTH,STAFF,IB_REGISTER_DATE,EB_REGISTER_CHANNEL,SMS,VERIFY_METHOD,AGE,Occupation_Group,Education_Level,Marital_Status,COUNT_CA_ACCT,AVG_CA_BALANCE,COUNT_TD_ACCT,AVG_TD_BALANCE,COUNT_OF_LOAN,AVG_LOAN_AMOUNT,OVERDUE_LENDING,TERM_LENDING,INTEREST_RATE,COUNT_CREDITCARD,COUNT_DEBITCARD,OVERDUE_CREDIT,LIMIT_AMT_CREDIT,OUTSTANDING_BAL_CREDIT,TRANS_RECORDS,TRANS_NO_SUM,TRANS_AMOUNT_SUM,TRANS_AMOUNT_MEAN,TRANS_AMOUNT_MAX,TRANS_ACTIVE_DAYS,TRANS_TYPE_LV1_NUNIQUE,TRANS_TYPE_LV2_NUNIQUE,DEVICE_NUNIQUE,MERCHANT_NUNIQUE,INTERNAL_TRANSFER_ROWS,ACTIVITY_RECORDS,ACTIVITY_NO_SUM,ACTIVITY_ACTIVE_DAYS,ACTIVITY_NAME_NUNIQUE,ACTIVITY_HOUR_NUNIQUE,ACTIVITY_TYPE_COUNT_ACCOUNT_ADDRESS_BOOK_DELETE,ACTIVITY_TYPE_COUNT_ACCOUNT_ADDRESS_BOOK_UPDATE,ACTIVITY_TYPE_COUNT_AUTHENTICATION,ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_ANNUALFEE,ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_CASHBACK,ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_REDEEM,ACTIVITY_TYPE_COUNT_CHANGE_PASSWORD,ACTIVITY_TYPE_COUNT_EXPORT_ACCOUNT_STATEMENT_LOAN,ACTIVITY_TYPE_COUNT_LOGIN,ACTIVITY_TYPE_COUNT_LOGIN_FACEID,ACTIVITY_TYPE_COUNT_LOGIN_FINGER,ACTIVITY_TYPE_COUNT_LOGOUT,ACTIVITY_TYPE_COUNT_MB_ACCOUNT_QUICK_BALANCE,ACTIVITY_TYPE_COUNT_MB_BILLPAY,ACTIVITY_TYPE_COUNT_MB_CHANGE_PIN,ACTIVITY_TYPE_COUNT_MB_EXCHANGE_RATE_VIEW,ACTIVITY_TYPE_COUNT_MB_INTEREST_RATE_VIEW,ACTIVITY_TYPE_COUNT_MB_LOCATION_ATM_VIEW,ACTIVITY_TYPE_COUNT_MB_LOCATION_BRANCH_VIEW,ACTIVITY_TYPE_COUNT_MB_LOCATION_POS_VIEW,ACTIVITY_TYPE_COUNT_MB_RESET_PIN,ACTIVITY_TYPE_COUNT_MB_SET_PIN,ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_INFORMATION,ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_PORFOLIO,ACTIVITY_TYPE_COUNT_QUERY_CURRENT_ACCOUNT,ACTIVITY_TYPE_COUNT_QUERY_LOAN_ACCOUNT,ACTIVITY_TYPE_COUNT_QUERY_MM_ACCOUNT,ACTIVITY_TYPE_COUNT_RB_BILLPAY_ADSL,ACTIVITY_TYPE_COUNT_RB_BILLPAY_HOMEPHONE,ACTIVITY_TYPE_COUNT_RB_BILLPAY_INSURANCE,ACTIVITY_TYPE_COUNT_RB_BILLPAY_MOBILE,ACTIVITY_TYPE_COUNT_RB_BILLPAY_PSTN,ACTIVITY_TYPE_COUNT_RB_BILLPAY_TELECOMMUNICATIONS,ACTIVITY_TYPE_COUNT_RB_BILLPAY_WATER,ACTIVITY_TYPE_COUNT_SET_PASSWORD,ACTIVITY_TYPE_COUNT_TOPUP_MOBILE,ACTIVITY_TYPE_COUNT_TRANSACTION_DETAIL_QUERY,ACTIVITY_TYPE_COUNT_TRANSACTION_OVERVIEW_QUERY,ACTIVITY_TYPE_COUNT_TRANSFER_INTERNATIONAL,ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PAYMENT_CENTER,ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PHONENO,ACTIVITY_TYPE_COUNT_TRANSFER_VIA_SML,ACTIVITY_TYPE_COUNT_TRANSFER_VIA_SML_ACCOUNT,ACTIVITY_TYPE_COUNT_TRANSFER_BANK_ACCOUNT,ACTIVITY_TYPE_COUNT_TRANSFER_BANK_ACCOUNT_BULK
0,0,2019-09-30,F,2019-09-16,1988-10-08,N,2019-09-17,BRANCH,N,SMART_OTP,38.0,Working,Incomplete higher,Married,1.0,57364833.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,99800000.0,9.980000e+07,99800000.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,10.0,1.0,7.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0,2019-10-31,F,2019-09-16,1988-10-08,N,2019-09-17,BRANCH,N,SMART_OTP,38.0,Working,Incomplete higher,Married,1.0,750812.90,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,2000000.0,2.000000e+06,2000000.0,1.0,1.0,1.0,1.0,1.0,1.0,15.0,29.0,4.0,6.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0,2019-11-30,F,2019-09-16,1988-10-08,N,2019-09-17,BRANCH,N,SMART_OTP,38.0,Working,Incomplete higher,Married,1.0,5280833.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,3.0,76000000.0,3.800000e+07,74000000.0,2.0,1.0,1.0,1.0,1.0,2.0,17.0,38.0,2.0,6.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
3,0,2019-12-31,F,2019-09-16,1988-10-08,N,2019-09-17,BRANCH,N,SMART_OTP,38.0,Working,Incomplete higher,Married,1.0,29460361.29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0

## 3. Feature Engineering From EDA

In [13]:
feature_df = add_customer_month_features(base)
print(feature_df.shape)
display(feature_df[['CUSTOMER_NUMBER','MONTH','SPTC_COUNT','TOTAL_DEPOSIT_BALANCE','TRANS_RECORDS','ACTIVITY_RECORDS','CUSTOMER_TENURE_MONTHS','IB_TENURE_MONTHS']].head())

(889631, 246)


,CUSTOMER_NUMBER,MONTH,SPTC_COUNT,TOTAL_DEPOSIT_BALANCE,TRANS_RECORDS,ACTIVITY_RECORDS,CUSTOMER_TENURE_MONTHS,IB_TENURE_MONTHS
0,0,2019-09-30,2,57364833.33,1.0,7.0,0.459959,0.427105
1,0,2019-10-31,2,750812.90,1.0,15.0,1.478439,1.445585
2,0,2019-11-30,2,5280833.33,2.0,17.0,2.464066,2.431211
3,0,2019-12-31,2,29460361.29,6.0,24.0,3.482546,3.449692
4,3,2019-04-30,1,0.00,0.0,0.0,0.459959,0.000000


## 4. Build Subscription Long Dataset

Target logic is aligned with `GCON_EDA_Subscription_Inline.ipynb`, including the first-subscription filter.

In [14]:
model_df = make_subscription_long(feature_df)

print(model_df.shape)
display(model_df.groupby('PRODUCT_NAME')['SUBSCRIPTION'].agg(['count', 'sum', 'mean']).sort_values('mean', ascending=False))
display(model_df[['CUSTOMER_NUMBER','PRODUCT_CODE','PRODUCT_NAME','MONTH','OWNS_PRODUCT_T','SUBSCRIPTION','CUM_SUBSCRIPTION_BEFORE','SPTC_COUNT']].head(20))

pre first-sub filter rows/positives: 1825865 87776
post first-sub filter rows/positives: 1795423 63545
(1795423, 265)


,count,sum,mean
PRODUCT_NAME,,,
CURRENT_ACCOUNT,130294,28664,0.219995
LENDING,354614,14881,0.041964
CREDIT_CARD,466199,9147,0.019620
DEBIT_CARD,307674,4960,0.016121
TERM_DEPOSIT,536642,5893,0.010981


,CUSTOMER_NUMBER,PRODUCT_CODE,PRODUCT_NAME,MONTH,OWNS_PRODUCT_T,SUBSCRIPTION,CUM_SUBSCRIPTION_BEFORE,SPTC_COUNT
0,0,103,CREDIT_CARD,2019-09-30,0,0,0,2
1,0,103,CREDIT_CARD,2019-10-31,0,0,0,2
2,0,105,LENDING,2019-09-30,0,0,0,2
3,0,105,LENDING,2019-10-31,0,0,0,2
4,0,102,TERM_DEPOSIT,2019-09-30,0,0,0,2
5,0,102,TERM_DEPOSIT,2019-10-31,0,0,0,2
6,3,103,CREDIT_CARD,2019-04-30,0,0,0,1
7,3,103,CREDIT_CARD,2019-05-31,0,0,0,2
8,3,103,CREDIT_CARD,2019-06-30,0,0,0,2
9,3,103,CREDIT_CARD,2019-07-31,0,0,0,2


## 5. Build Model-Ready Matrix

In [15]:
product_ohe = pd.get_dummies(model_df['PRODUCT_NAME'], prefix='PREDICT_PRODUCT', dtype='int8')
model_df = pd.concat([model_df, product_ohe], axis=1)

id_cols = ['CUSTOMER_NUMBER', 'PRODUCT_CODE', 'PRODUCT_NAME', 'MONTH']
target_col = 'SUBSCRIPTION'

manual_drop_features = [
    'AGE', 'HAS_IB',
    'OWNS_PRODUCT_T', 'FUTURE_OWNS_PRODUCT', 'HAS_FULL_LABEL_HORIZON', 'CUM_SUBSCRIPTION_BEFORE',
    'CLIENT_CREATE_DATE', 'DATE_OF_BIRTH', 'IB_REGISTER_DATE',
]
exclude_cols = set(id_cols + [target_col, 'OWN_COL'] + manual_drop_features)
date_cols = [c for c in model_df.columns if pd.api.types.is_datetime64_any_dtype(model_df[c]) and c not in ['MONTH']]
future_label_cols = [c for c in model_df.columns if c.startswith('OWNS_PRODUCT_T_PLUS_')]
exclude_cols.update(date_cols)
exclude_cols.update(future_label_cols)

candidate_cols = [c for c in model_df.columns if c not in exclude_cols]
cat_cols = [c for c in candidate_cols if str(model_df[c].dtype) in ['object', 'string', 'category']]
num_cols = [c for c in candidate_cols if c not in cat_cols]

model_ready = pd.concat([
    model_df[id_cols + [target_col]].reset_index(drop=True),
    model_df[num_cols].reset_index(drop=True),
    pd.get_dummies(model_df[cat_cols].astype('string').fillna('UNKNOWN'), dummy_na=False, dtype='int8').reset_index(drop=True),
], axis=1)

# XGBoost rejects feature names containing [, ], or <. Sanitize only model feature columns.
def sanitize_feature_name(name):
    name = str(name)
    for bad, repl in {'[': '(', ']': ')', '<': 'lt_', '>': 'gt_', '=': 'eq_'}.items():
        name = name.replace(bad, repl)
    return name.replace(' ', '_').replace(',', '_').replace(';', '_')

rename_map = {}
used_names = set(id_cols + [target_col])
for col in model_ready.columns:
    if col in id_cols + [target_col]:
        continue
    clean = sanitize_feature_name(col)
    base_clean = clean
    counter = 1
    while clean in used_names:
        counter += 1
        clean = f'{base_clean}_{counter}'
    rename_map[col] = clean
    used_names.add(clean)
model_ready = model_ready.rename(columns=rename_map)

model_ready = model_ready.replace([np.inf, -np.inf], np.nan)
feature_cols = [c for c in model_ready.columns if c not in id_cols + [target_col]]
model_ready[feature_cols] = model_ready[feature_cols].fillna(0)

# Keep all numeric dtypes compact enough for tree models.
for col in feature_cols:
    if model_ready[col].dtype == 'float64':
        model_ready[col] = model_ready[col].astype('float32')
    elif model_ready[col].dtype == 'int64':
        model_ready[col] = pd.to_numeric(model_ready[col], downcast='integer')

if USE_SAMPLE_FOR_FAST_RUN and len(model_ready) > SAMPLE_ROWS:
    model_ready = model_ready.sample(SAMPLE_ROWS, random_state=RANDOM_STATE).sort_values(['MONTH','CUSTOMER_NUMBER']).reset_index(drop=True)

print('model_ready:', model_ready.shape)
print('features:', len(feature_cols))
print('categorical source columns:', cat_cols)
display(model_ready.head())

model_ready: (1795423, 293)
features: 288
categorical source columns: ['CLIENT_SEX', 'STAFF', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'Occupation_Group', 'Education_Level', 'Marital_Status', 'AGE_GROUP', 'CUSTOMER_TENURE_BIN', 'ACTIVITY_SEGMENT']


,CUSTOMER_NUMBER,PRODUCT_CODE,PRODUCT_NAME,MONTH,SUBSCRIPTION,COUNT_CA_ACCT,AVG_CA_BALANCE,COUNT_TD_ACCT,AVG_TD_BALANCE,COUNT_OF_LOAN,AVG_LOAN_AMOUNT,OVERDUE_LENDING,TERM_LENDING,INTEREST_RATE,COUNT_CREDITCARD,COUNT_DEBITCARD,OVERDUE_CREDIT,LIMIT_AMT_CREDIT,OUTSTANDING_BAL_CREDIT,TRANS_RECORDS,TRANS_NO_SUM,TRANS_AMOUNT_SUM,TRANS_AMOUNT_MEAN,TRANS_AMOUNT_MAX,TRANS_ACTIVE_DAYS,TRANS_TYPE_LV1_NUNIQUE,TRANS_TYPE_LV2_NUNIQUE,DEVICE_NUNIQUE,MERCHANT_NUNIQUE,INTERNAL_TRANSFER_ROWS,ACTIVITY_RECORDS,ACTIVITY_NO_SUM,ACTIVITY_ACTIVE_DAYS,ACTIVITY_NAME_NUNIQUE,ACTIVITY_HOUR_NUNIQUE,ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_CASHBACK,ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_REDEEM,ACTIVITY_TYPE_COUNT_EXPORT_ACCOUNT_STATEMENT_LOAN,ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_INFORMATION,ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_PORFOLIO,ACTIVITY_TYPE_COUNT_QUERY_CURRENT_ACCOUNT,ACTIVITY_TYPE_COUNT_RB_BILLPAY_MOBILE,ACTIVITY_TYPE_COUNT_TOPUP_MOBILE,ACTIVITY_TYPE_COUNT_TRANSACTION_DETAIL_QUERY,ACTIVITY_TYPE_COUNT_TRANSACTION_OVERVIEW_QUERY,ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PAYMENT_CENTER,ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PHONENO,ACTIVITY_TYPE_COUNT_TRANSFER_BANK_ACCOUNT,OWN_CURRENT_ACCOUNT,OWN_TERM_DEPOSIT,OWN_CREDIT_CARD,OWN_DEBIT_CARD,OWN_LENDING,SPTC_COUNT,HAS_MULTI_SPTC,TOTAL_DEPOSIT_BALANCE,TOTAL_DEPOSIT_ACCTS,CARD_UTILIZATION,HAS_TRANSACTION,HAS_ACTIVITY,ACTIVITY_EVENTS_PER_ACTIVE_DAY,ACTIVITY_TYPES_PER_EVENT,ACTIVITY_HOURS_PER_ACTIVE_DAY,AGE_CLEAN,CUSTOMER_TENURE_MONTHS,IB_TENURE_MONTHS,LOG_TOTAL_DEPOSIT_BALANCE,LOG_AVG_CA_BALANCE,LOG_AVG_TD_BALANCE,LOG_AVG_LOAN_AMOUNT,LOG_TRANS_AMOUNT_SUM,LOG_TRANS_AMOUNT_MEAN,LOG_TRANS_AMOUNT_MAX,LOG_ACTIVITY_NO_SUM,TOTAL_DEPOSIT_BALANCE_LAG1,TOTAL_DEPOSIT_BALANCE_DIFF1,TOTAL_DEPOSIT_BALANCE_ROLL3_MEAN,TOTAL_DEPOSIT_ACCTS_LAG1,TOTAL_DEPOSIT_ACCTS_DIFF1,TOTAL_DEPOSIT_ACCTS_ROLL3_MEAN,TRANS_AMOUNT_SUM_LAG1,TRANS_AMOUNT_SUM_DIFF1,TRANS_AMOUNT_SUM_ROLL3_MEAN,TRANS_RECORDS_LAG1,TRANS_RECORDS_DIFF1,TRANS_RECORDS_ROLL3_MEAN,TRANS_ACTIVE_DAYS_LAG1,TRANS_ACTIVE_DAYS_DIFF1,TRANS_ACTIVE_DAYS_ROLL3_MEAN,ACTIVITY_RECORDS_LAG1,ACTIVITY_RECORDS_DIFF1,ACTIVITY_RECORDS_ROLL3_MEAN,ACTIVITY_ACTIVE_DAYS_LAG1,ACTIVITY_ACTIVE_DAYS_DIFF1,ACTIVITY_ACTIVE_DAYS_ROLL3_MEAN,ACTIVITY_NAME_NUNIQUE_LAG1,ACTIVITY_NAME_NUNIQUE_DIFF1,ACTIVITY_NAME_NUNIQUE_ROLL3_MEAN,SPTC_COUNT_LAG1,SPTC_COUNT_DIFF1,...,LENDING_OPEN_COUNT_HISTORY,LENDING_OWNED_MONTHS_HISTORY,LENDING_OWN_RATE_HISTORY,LENDING_COUNT_LAST_30D,LENDING_COUNT_LAST_90D,LENDING_COUNT_AVG_90D,PRODUCT_OPEN_COUNT_HISTORY,PRODUCT_TRANSITION_COUNT_LAST_90D,LOG_TXN_COUNT_LAST_90D,LOG_TOTAL_TRANSACTION_AMOUNT_90D,LOG_AVG_BALANCE_90D,LOG_DAYS_SINCE_LAST_TRANSACTION,LOG_DAYS_SINCE_LAST_LOGIN,LOG_DAYS_SINCE_LAST_PRODUCT_OPEN,LOG_ACTIVITY_COUNT_LAST_90D,LOG_ACTIVITY_ACTIVE_DAYS_LAST_90D,LOG_AVG_MONTHLY_ACTIVITY,LOG_ACTIVITY_RECORDS_PER_ACTIVE_DAY_90D,LOG_ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_INFORMATION_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_QUERY_ACCOUNT_PORFOLIO_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_QUERY_CURRENT_ACCOUNT_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_TRANSACTION_DETAIL_QUERY_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_TRANSACTION_OVERVIEW_QUERY_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_TRANSFER_BANK_ACCOUNT_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PHONENO_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_TRANSFER_VIA_PAYMENT_CENTER_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_RB_BILLPAY_MOBILE_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_TOPUP_MOBILE_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_CASHBACK_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_CARD_EGIFT_REGISTER_REDEEM_LAST_90D,LOG_ACTIVITY_TYPE_COUNT_EXPORT_ACCOUNT_STATEMENT_LOAN_LAST_90D,LOG_DAYS_SINCE_LAST_OPEN_CURRENT_ACCOUNT,LOG_CURRENT_ACCOUNT_COUNT_LAST_90D,LOG_DAYS_SINCE_LAST_OPEN_TERM_DEPOSIT,LOG_TERM_DEPOSIT_COUNT_LAST_90D,LOG_DAYS_SINCE_LAST_OPEN_CREDIT_CARD,LOG_CREDIT_CARD_COUNT_LAST_90D,LOG_DAYS_SINCE_LAST_OPEN_DEBIT_CARD,LOG_DEBIT_CARD_COUNT_LAST_90D,LOG_DAYS_SINCE_LAST_OPEN_LENDING,LOG_LENDING_COUNT_LAST_90D,TARGET_DAYS_SINCE_LAST_PRODUCT_OPEN,TARGET_PRODUCT_OPEN_COUNT_HISTORY,TARGET_PRODUCT_OWNED_MONTHS_HISTOR

## 6. Time-Based Train/Test Split

In [16]:
months = sorted(model_ready['MONTH'].dropna().unique())
if TEST_START_MONTH is None:
    test_start = months[-1]
else:
    test_start = pd.Timestamp(TEST_START_MONTH).to_period('M').to_timestamp('M')

train_df = model_ready[model_ready['MONTH'] < test_start].copy()
test_df = model_ready[model_ready['MONTH'] == test_start].copy()

# Product affinity / co-ownership features.
# Learn co-ownership matrix only from train-period customer-month rows to avoid test leakage.
own_cols = PRODUCTS['OWN_COL'].tolist()
co_base = feature_df.loc[feature_df['MONTH'] < test_start, ['CUSTOMER_NUMBER', 'MONTH'] + own_cols].drop_duplicates()
co_matrix = pd.DataFrame(index=own_cols, columns=own_cols, dtype='float32')
for source_col in own_cols:
    source_mask = co_base[source_col] == 1
    for target_col_name in own_cols:
        co_matrix.loc[source_col, target_col_name] = co_base.loc[source_mask, target_col_name].mean() if source_mask.any() else 0
co_matrix = co_matrix.fillna(0).astype('float32')
base_ownership_rate = co_base[own_cols].mean().to_dict()
product_to_own = PRODUCTS.set_index('PRODUCT_NAME')['OWN_COL'].to_dict()

print('train-period co-ownership matrix')
display(co_matrix)

co_feature_cols = [
    'PRODUCT_AFFINITY_SUM',
    'PRODUCT_AFFINITY_AVG',
    'PRODUCT_AFFINITY_MAX',
    'PRODUCT_COOCCURRENCE_BASE_RATE',
    'PRODUCT_COOCCURRENCE_OWNED_SOURCE_COUNT',
]

def add_product_affinity(df):
    df = df.copy()
    owned_values = df[own_cols].to_numpy(dtype='float32')
    affinity_sum = np.zeros(len(df), dtype='float32')
    affinity_max = np.zeros(len(df), dtype='float32')
    base_rate = np.zeros(len(df), dtype='float32')
    for product_name, target_own_col in product_to_own.items():
        mask = df['PRODUCT_NAME'].eq(product_name).to_numpy()
        if not mask.any():
            continue
        weights = co_matrix[target_own_col].reindex(own_cols).to_numpy(dtype='float32')
        vals = owned_values[mask] * weights
        affinity_sum[mask] = vals.sum(axis=1)
        affinity_max[mask] = vals.max(axis=1)
        base_rate[mask] = float(base_ownership_rate.get(target_own_col, 0))
    source_count = df[own_cols].sum(axis=1).astype('float32').to_numpy()
    df['PRODUCT_AFFINITY_SUM'] = affinity_sum
    df['PRODUCT_AFFINITY_AVG'] = affinity_sum / np.maximum(source_count, 1)
    df['PRODUCT_AFFINITY_MAX'] = affinity_max
    df['PRODUCT_COOCCURRENCE_BASE_RATE'] = base_rate
    df['PRODUCT_COOCCURRENCE_OWNED_SOURCE_COUNT'] = source_count
    return df

train_df = add_product_affinity(train_df)
test_df = add_product_affinity(test_df)
for col in co_feature_cols:
    if col not in feature_cols:
        feature_cols.append(col)

model_ready = pd.concat([train_df, test_df], ignore_index=True).sort_values(['MONTH', 'CUSTOMER_NUMBER', 'PRODUCT_CODE']).reset_index(drop=True)

print('test_start:', test_start)
print('train:', train_df.shape, train_df['MONTH'].min(), train_df['MONTH'].max(), 'positive_rate=', train_df[target_col].mean())
print('test :', test_df.shape, test_df['MONTH'].min(), test_df['MONTH'].max(), 'positive_rate=', test_df[target_col].mean())
display(train_df.groupby('PRODUCT_NAME')[target_col].agg(['count','sum','mean']))
display(test_df.groupby('PRODUCT_NAME')[target_col].agg(['count','sum','mean']))

train-period co-ownership matrix


,OWN_CURRENT_ACCOUNT,OWN_TERM_DEPOSIT,OWN_CREDIT_CARD,OWN_DEBIT_CARD,OWN_LENDING
OWN_CURRENT_ACCOUNT,1.000000,0.072201,0.091003,0.541668,0.385609
OWN_TERM_DEPOSIT,0.616704,1.000000,0.132566,0.258773,0.134702
OWN_CREDIT_CARD,0.335873,0.057283,1.000000,0.074343,0.741987
OWN_DEBIT_CARD,0.875041,0.048942,0.032540,1.000000,0.094833
OWN_LENDING,0.749572,0.030655,0.390789,0.114112,1.000000


test_start: 2019-10-31 00:00:00
train: (1423922, 298) 2019-01-31 00:00:00 2019-09-30 00:00:00 positive_rate= 0.03619510057432921
test : (371501, 298) 2019-10-31 00:00:00 2019-10-31 00:00:00 positive_rate= 0.03231754423272077


,count,sum,mean
PRODUCT_NAME,,,
CREDIT_CARD,369900,7265,0.019640
CURRENT_ACCOUNT,104840,22486,0.214479
DEBIT_CARD,246554,4214,0.017092
LENDING,278192,12657,0.045497
TERM_DEPOSIT,424436,4917,0.011585


,count,sum,mean
PRODUCT_NAME,,,
CREDIT_CARD,96299,1882,0.019543
CURRENT_ACCOUNT,25454,6178,0.242712
DEBIT_CARD,61120,746,0.012205
LENDING,76422,2224,0.029102
TERM_DEPOSIT,112206,976,0.008698


## 7. Save Model Input Tables

In [17]:
save_table(model_df, 'gcon_subscription_long_h2_raw_features')
save_table(model_ready, 'gcon_model_ready_h2')
save_table(train_df, 'gcon_train_h2')
save_table(test_df, 'gcon_test_h2')

feature_meta = pd.DataFrame({'feature': feature_cols})
safe_to_csv(feature_meta, OUTPUT_DIR / 'gcon_h2_feature_columns.csv')
safe_to_csv(PRODUCTS, OUTPUT_DIR / 'gcon_product_mapping.csv')

saved model_data_gcon\gcon_subscription_long_h2_raw_features.parquet (1795423, 270)
saved model_data_gcon\gcon_model_ready_h2.parquet (1795423, 298)
saved model_data_gcon\gcon_train_h2.parquet (1423922, 298)
saved model_data_gcon\gcon_test_h2.parquet (371501, 298)
saved model_data_gcon\gcon_h2_feature_columns.csv (293, 1)
saved model_data_gcon\gcon_product_mapping.csv (5, 3)


## 8. Train LightGBM and XGBoost

In [18]:
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

X_train = train_df[feature_cols]
y_train = train_df[target_col].astype(int)
X_test = test_df[feature_cols]
y_test = test_df[target_col].astype(int)

neg = max(int((y_train == 0).sum()), 1)
pos = max(int((y_train == 1).sum()), 1)
scale_pos_weight = neg / pos
print('scale_pos_weight:', scale_pos_weight)

models = {}

from lightgbm import LGBMClassifier
models['lightgbm'] = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.035,
    num_leaves=31,
    max_depth=-1,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

from xgboost import XGBClassifier
models['xgboost'] = XGBClassifier(
    n_estimators=350,
    learning_rate=0.04,
    max_depth=5,
    min_child_weight=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model_scores = {}
fit_rows = []
for name, model in models.items():
    start = time.time()
    print('training', name)
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    score = model.predict_proba(X_test)[:, 1]
    model_scores[name] = score
    fit_rows.append({'model': name, 'fit_seconds': elapsed})
    print(name, 'fit_seconds=', round(elapsed, 1), 'score_mean=', score.mean())

fit_summary = pd.DataFrame(fit_rows)
display(fit_summary)

scale_pos_weight: 26.628048662178156
training lightgbm
lightgbm fit_seconds= 73.4 score_mean= 0.1933268557038027
training xgboost
xgboost fit_seconds= 125.3 score_mean= 0.19926153


,model,fit_seconds
0,lightgbm,73.399597
1,xgboost,125.251215


## 9. Compare Model Metrics

In [19]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

metric_rows = []
threshold_rows = []
ranking_rows = []
product_metric_rows = []
scored_frames = []

for name, score in model_scores.items():
    y_score = pd.Series(score, index=test_df.index, name='SUBSCRIPTION_PROPENSITY')
    y_true = y_test.reset_index(drop=True)
    clipped = np.clip(score, 1e-6, 1 - 1e-6)
    row = {
        'model': name,
        'rows': len(test_df),
        'positive_rate': float(y_test.mean()),
        'avg_score': float(np.mean(score)),
        'log_loss': float(log_loss(y_test, clipped)),
    }
    if y_test.nunique() == 2:
        row['roc_auc'] = float(roc_auc_score(y_test, score))
        row['pr_auc_average_precision'] = float(average_precision_score(y_test, score))
    metric_rows.append(row)

    for threshold in [0.05, 0.10, 0.20, 0.30, 0.50]:
        y_pred = (score >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
        threshold_rows.append({
            'model': name,
            'threshold': threshold,
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        })

    scored = test_df[id_cols + [target_col]].copy()
    scored['SUBSCRIPTION_PROPENSITY'] = score
    scored['MODEL'] = name
    scored_frames.append(scored)

    ranked = scored.sort_values(
        ['CUSTOMER_NUMBER', 'MONTH', 'SUBSCRIPTION_PROPENSITY'],
        ascending=[True, True, False],
    ).copy()
    ranked['RANK_IN_CUSTOMER_MONTH'] = ranked.groupby(['CUSTOMER_NUMBER','MONTH']).cumcount() + 1
    for k in [1, 2, 3]:
        topk = ranked.query('RANK_IN_CUSTOMER_MONTH <= @k')
        group_hit = topk.groupby(['CUSTOMER_NUMBER','MONTH'])[target_col].max()
        ranking_rows.append({
            'model': name,
            'k': k,
            'hit_rate_at_k': float(group_hit.mean()),
            'precision_at_k': float(topk[target_col].mean()),
            'positive_captured': int(topk[target_col].sum()),
            'rows_recommended': len(topk),
        })

    for product_name, g in scored.groupby('PRODUCT_NAME'):
        prow = {
            'model': name,
            'PRODUCT_NAME': product_name,
            'rows': len(g),
            'actual_positive_rate': float(g[target_col].mean()),
            'avg_predicted_propensity': float(g['SUBSCRIPTION_PROPENSITY'].mean()),
        }
        if g[target_col].nunique() == 2:
            prow['roc_auc'] = float(roc_auc_score(g[target_col], g['SUBSCRIPTION_PROPENSITY']))
            prow['pr_auc'] = float(average_precision_score(g[target_col], g['SUBSCRIPTION_PROPENSITY']))
        product_metric_rows.append(prow)

comparison_metrics = pd.DataFrame(metric_rows).merge(fit_summary, on='model', how='left').sort_values('pr_auc_average_precision', ascending=False)
threshold_metrics = pd.DataFrame(threshold_rows)
ranking_metrics = pd.DataFrame(ranking_rows)
product_metrics = pd.DataFrame(product_metric_rows)
test_scored_all_models = pd.concat(scored_frames, ignore_index=True)

display(comparison_metrics)
display(ranking_metrics.sort_values(['k','hit_rate_at_k'], ascending=[True, False]))
display(product_metrics.sort_values(['PRODUCT_NAME','pr_auc'], ascending=[True, False]))

,model,rows,positive_rate,avg_score,log_loss,roc_auc,pr_auc_average_precision,fit_seconds
1,xgboost,371501,0.032318,0.199262,0.250602,0.949173,0.704676,125.251215
0,lightgbm,371501,0.032318,0.193327,0.244411,0.950176,0.699992,73.399597


,model,k,hit_rate_at_k,precision_at_k,positive_captured,rows_recommended
0,lightgbm,1,0.080528,0.080528,10070,125049
3,xgboost,1,0.080384,0.080384,10052,125049
4,xgboost,2,0.086622,0.045853,11294,246308
1,lightgbm,2,0.086598,0.045938,11315,246308
2,lightgbm,3,0.088269,0.033864,11859,350191
5,xgboost,3,0.088237,0.033842,11851,350191


,model,PRODUCT_NAME,rows,actual_positive_rate,avg_predicted_propensity,roc_auc,pr_auc
0,lightgbm,CREDIT_CARD,96299,0.019543,0.203151,0.889401,0.197651
5,xgboost,CREDIT_CARD,96299,0.019543,0.210187,0.886285,0.184457
6,xgboost,CURRENT_ACCOUNT,25454,0.242712,0.478334,0.979181,0.943570
1,lightgbm,CURRENT_ACCOUNT,25454,0.242712,0.470462,0.978205,0.930451
7,xgboost,DEBIT_CARD,61120,0.012205,0.166842,0.882269,0.231123
2,lightgbm,DEBIT_CARD,61120,0.012205,0.160036,0.884734,0.229576
8,xgboost,LENDING,76422,0.029102,0.194102,0.937361,0.558785
3,lightgbm,LENDING,76422,0.029102,0.190296,0.938134,0.557868
9,xgboost,TERM_DEPOSIT,112206,0.008698,0.147751,0.878961,0.098675
4,lightgbm,TERM_DEPOSIT,112206,0.008698,0.142226,0.880975,0.090111


## 10. Save Scores, Metrics, Recommendations

In [20]:
safe_to_csv(comparison_metrics, OUTPUT_DIR / 'gcon_h2_model_comparison_metrics.csv')
safe_to_csv(threshold_metrics, OUTPUT_DIR / 'gcon_h2_threshold_metrics.csv')
safe_to_csv(ranking_metrics, OUTPUT_DIR / 'gcon_h2_ranking_metrics.csv')
safe_to_csv(product_metrics, OUTPUT_DIR / 'gcon_h2_product_metrics.csv')
save_table(test_scored_all_models, 'gcon_test_scored_h2_all_models')

best_model_name = comparison_metrics.iloc[0]['model']
best_scored = test_scored_all_models.query('MODEL == @best_model_name').copy()
recommendations = (
    best_scored
    .sort_values(['CUSTOMER_NUMBER','MONTH','SUBSCRIPTION_PROPENSITY'], ascending=[True, True, False])
    .groupby(['CUSTOMER_NUMBER','MONTH'], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
save_table(recommendations, f'gcon_recommendations_h2_best_{best_model_name}')
print('best model:', best_model_name)
display(recommendations.head())

saved model_data_gcon\gcon_h2_model_comparison_metrics.csv (2, 8)
saved model_data_gcon\gcon_h2_threshold_metrics.csv (10, 9)
saved model_data_gcon\gcon_h2_ranking_metrics.csv (6, 6)
saved model_data_gcon\gcon_h2_product_metrics.csv (10, 7)
saved model_data_gcon\gcon_test_scored_h2_all_models.parquet (743002, 7)
saved model_data_gcon\gcon_recommendations_h2_best_xgboost.parquet (125049, 7)
best model: xgboost


,CUSTOMER_NUMBER,PRODUCT_CODE,PRODUCT_NAME,MONTH,SUBSCRIPTION,SUBSCRIPTION_PROPENSITY,MODEL
0,0,102,TERM_DEPOSIT,2019-10-31,0,0.424461,xgboost
1,3,102,TERM_DEPOSIT,2019-10-31,0,0.058242,xgboost
2,9,103,CREDIT_CARD,2019-10-31,0,0.720658,xgboost
3,13,101,CURRENT_ACCOUNT,2019-10-31,0,0.912762,xgboost
4,14,101,CURRENT_ACCOUNT,2019-10-31,0,0.111137,xgboost


## 11. Feature Importance

In [21]:
importance_frames = []
for name, model in models.items():
    if hasattr(model, 'feature_importances_'):
        imp = pd.DataFrame({
            'model': name,
            'feature': feature_cols,
            'importance': model.feature_importances_,
        }).sort_values('importance', ascending=False)
        importance_frames.append(imp)

feature_importance = pd.concat(importance_frames, ignore_index=True) if importance_frames else pd.DataFrame()
safe_to_csv(feature_importance, OUTPUT_DIR / 'gcon_h2_feature_importance.csv')
display(feature_importance.groupby('model').head(30))

saved model_data_gcon\gcon_h2_feature_importance.csv (586, 3)


,model,feature,importance
0,lightgbm,PRODUCT_AFFINITY_AVG,648.000000
1,lightgbm,AGE_CLEAN,507.000000
2,lightgbm,CUSTOMER_TENURE_MONTHS,442.000000
3,lightgbm,IB_TENURE_MONTHS,429.000000
4,lightgbm,INTEREST_RATE,406.000000
5,lightgbm,AVG_LOAN_AMOUNT,371.000000
6,lightgbm,PRODUCT_AFFINITY_SUM,286.000000
7,lightgbm,LIMIT_AMT_CREDIT,253.000000
8,lightgbm,AVG_BALANCE_90D,242.000000
9,lightgbm,PREDICT_PRODUCT_DEBIT_CARD,238.000000
